In [1]:
import pandas as pd
import numpy as np

In [3]:
file_path = "online_retail_II.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name="Year 2009-2010"
)

print("Dataset Shape:")
print(df.shape)

print("\nFirst 5 Rows:")
print(df.head())

Dataset Shape:
(525461, 8)

First 5 Rows:
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

          InvoiceDate  Price  Customer ID         Country  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3 2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4 2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [4]:
print("Columns:")
print(df.columns.tolist())

Columns:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


In [5]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64


In [6]:
df = df.dropna(subset=["Customer ID"])

print("Dataset Shape After Removing Missing Customer IDs:")
print(df.shape)

Dataset Shape After Removing Missing Customer IDs:
(417534, 8)


In [7]:
df = df[~df["Invoice"].astype(str).str.startswith("C")]

print("Dataset Shape After Removing Cancellations:")
print(df.shape)

Dataset Shape After Removing Cancellations:
(407695, 8)


In [8]:
print("Quantity <= 0:")
print((df["Quantity"] <= 0).sum())

print("\nQuantity statistics:")
print(df["Quantity"].describe())

Quantity <= 0:
0

Quantity statistics:
count    407695.000000
mean         13.586686
std          96.842229
min           1.000000
25%           2.000000
50%           5.000000
75%          12.000000
max       19152.000000
Name: Quantity, dtype: float64


In [9]:
df = df.dropna(subset=["Description"])

print("Dataset Shape After Removing Missing Descriptions:")
print(df.shape)

print("\nMissing Descriptions:")
print(df["Description"].isnull().sum())

Dataset Shape After Removing Missing Descriptions:
(407695, 8)

Missing Descriptions:
0


In [10]:
# Check duplicate product entries within the same invoice

duplicates = df.duplicated(subset=["Invoice", "StockCode"]).sum()

print("Duplicate product entries within invoices:", duplicates)

Duplicate product entries within invoices: 12708


In [11]:
# Keep only one occurrence of each product per invoice

df = df.drop_duplicates(subset=["Invoice", "StockCode"])

print("Dataset Shape After Removing Duplicate Products:")
print(df.shape)

print("\nRemaining Duplicate Product Entries:")
print(df.duplicated(subset=["Invoice", "StockCode"]).sum())

Dataset Shape After Removing Duplicate Products:
(394987, 8)

Remaining Duplicate Product Entries:
0


In [12]:
# Create a list of products purchased in each invoice

basket = df.groupby("Invoice")["Description"].apply(list)

print("Number of Invoices:", len(basket))

print("\nExample of products in the first invoice:")
print(basket.iloc[0])

Number of Invoices: 19215

Example of products in the first invoice:
['15CM CHRISTMAS GLASS BALL 20 LIGHTS', 'PINK CHERRY LIGHTS', ' WHITE CHERRY LIGHTS', 'RECORD FRAME 7" SINGLE SIZE ', 'STRAWBERRY CERAMIC TRINKET BOX', 'PINK DOUGHNUT TRINKET POT ', 'SAVE THE PLANET MUG', 'FANCY FONT HOME SWEET HOME DOORMAT']


In [13]:
from itertools import combinations
from collections import Counter

# Generate product pairs from each invoice
pair_counts = Counter()

for products in basket:
    for pair in combinations(sorted(set(products)), 2):
        pair_counts[pair] += 1

print("Total Unique Product Pairs:", len(pair_counts))

Total Unique Product Pairs: 2363449


In [14]:
# Convert pair counts into a DataFrame

top_pairs = pd.DataFrame(
    pair_counts.items(),
    columns=["Product Pair", "Purchase Count"]
)

# Sort by purchase frequency
top_pairs = top_pairs.sort_values(
    by="Purchase Count",
    ascending=False
).reset_index(drop=True)

# Display top 20 pairs
print("Top 20 Product Pairs:")
print(top_pairs.head(20))

Top 20 Product Pairs:
                                         Product Pair  Purchase Count
0   (RED HANGING HEART T-LIGHT HOLDER, WHITE HANGI...             722
1   (STRAWBERRY CERAMIC TRINKET BOX, SWEETHEART CE...             622
2   (WOODEN FRAME ANTIQUE WHITE , WOODEN PICTURE F...             554
3   (HOME BUILDING BLOCK WORD, LOVE BUILDING BLOCK...             522
4      (HEART OF WICKER LARGE, HEART OF WICKER SMALL)             491
5   (60 TEATIME FAIRY CAKE CASES, PACK OF 72 RETRO...             478
6   (60 TEATIME FAIRY CAKE CASES, PACK OF 60 PINK ...             468
7   (LUNCH BAG SPACEBOY DESIGN , LUNCH BAG SUKI  D...             454
8   (WHITE HANGING HEART T-LIGHT HOLDER, WOODEN FR...             442
9   (PACK OF 60 PINK PAISLEY CAKE CASES, PACK OF 7...             409
10    (LUNCH BAG  BLACK SKULL., LUNCH BAG RED SPOTTY)             403
11   (LUNCH BAG PINK RETROSPOT, LUNCH BAG RED SPOTTY)             394
12  (HOME BUILDING BLOCK WORD, WHITE HANGING HEART...             39

In [15]:
# Create a clean Top 10 Product Pairs table

top_10_pairs = top_pairs.head(10).copy()

top_10_pairs["Product 1"] = top_10_pairs["Product Pair"].apply(lambda x: x[0])
top_10_pairs["Product 2"] = top_10_pairs["Product Pair"].apply(lambda x: x[1])

top_10_pairs = top_10_pairs[
    ["Product 1", "Product 2", "Purchase Count"]
]

print("Top 10 Product Pairs:")
display(top_10_pairs)

Top 10 Product Pairs:


,Product 1,Product 2,Purchase Count
0,RED HANGING HEART T-LIGHT HOLDER,WHITE HANGING HEART T-LIGHT HOLDER,722
1,STRAWBERRY CERAMIC TRINKET BOX,SWEETHEART CERAMIC TRINKET BOX,622
2,WOODEN FRAME ANTIQUE WHITE,WOODEN PICTURE FRAME WHITE FINISH,554
3,HOME BUILDING BLOCK WORD,LOVE BUILDING BLOCK WORD,522
4,HEART OF WICKER LARGE,HEART OF WICKER SMALL,491
5,60 TEATIME FAIRY CAKE CASES,PACK OF 72 RETRO SPOT CAKE CASES,478
6,60 TEATIME FAIRY CAKE CASES,PACK OF 60 PINK PAISLEY CAKE CASES,468
7,LUNCH BAG SPACEBOY DESIGN,LUNCH BAG SUKI DESIGN,454
8,WHITE HANGING HEART T-LIGHT HOLDER,WOODEN FRAME ANTIQUE WHITE,442
9,PACK OF 60 PINK PAISLEY CAKE CASES,PACK OF 72 RETRO SPOT CAKE CASES,409


In [16]:
recommendations = [
    "Create bundle offers for products frequently purchased together.",
    "Use cross-selling recommendations at checkout based on common product pairs.",
    "Place frequently paired products near each other in the online store.",
    "Offer discounts on complementary products to increase basket size.",
    "Promote popular product combinations through email and marketing campaigns."
]

print("Recommendations:")
for i, recommendation in enumerate(recommendations, 1):
    print(f"{i}. {recommendation}")

Recommendations:
1. Create bundle offers for products frequently purchased together.
2. Use cross-selling recommendations at checkout based on common product pairs.
3. Place frequently paired products near each other in the online store.
4. Offer discounts on complementary products to increase basket size.
5. Promote popular product combinations through email and marketing campaigns.


In [17]:
print("TASK 29: PRODUCT BASKET ANALYSIS")
print("=" * 50)

print("\nDataset Summary:")
print(f"Total Invoices Analyzed: {len(basket):,}")
print(f"Unique Product Pairs Found: {len(pair_counts):,}")

print("\nTop Product Pair:")
print(f"{top_10_pairs.iloc[0]['Product 1']} + "
      f"{top_10_pairs.iloc[0]['Product 2']}")
print(f"Purchased Together: {top_10_pairs.iloc[0]['Purchase Count']:,} times")

print("\nKey Recommendation:")
print("Use frequently purchased product pairs for bundle offers,")
print("cross-selling, product placement, and targeted promotions.")

TASK 29: PRODUCT BASKET ANALYSIS

Dataset Summary:
Total Invoices Analyzed: 19,215
Unique Product Pairs Found: 2,363,449

Top Product Pair:
RED HANGING HEART T-LIGHT HOLDER + WHITE HANGING HEART T-LIGHT HOLDER
Purchased Together: 722 times

Key Recommendation:
Use frequently purchased product pairs for bundle offers,
cross-selling, product placement, and targeted promotions.
